In [ ]:
import os

# ==========================================
# CAPA DE DATOS (Data Layer)
# ==========================================
class CapaDatos:
    def __init__(self, nombreArchivo):
        self.nombreArchivo = nombreArchivo

    def guardarRegistro(self, cadenaRegistro):
        with open(self.nombreArchivo, "a", encoding="utf-8") as archivo:
            archivo.write(cadenaRegistro + "\n")

    def leerTodosLosRegistros(self):
        if not os.path.exists(self.nombreArchivo):
            return []
        with open(self.nombreArchivo, "r", encoding="utf-8") as archivo:
            return archivo.readlines()

# ==========================================
# CAPA DE NEGOCIO (Business Logic Layer)
# ==========================================
class CapaNegocio:
    def __init__(self):
        self.capaDatos = CapaDatos("inspecciones.txt")

    def agregarInspeccion(self, idInspeccion, datosConglomerado, datosBrigada, conteoFustales, conteoLatizales, conteoBrinzales, listaMuestras):
        cadenaMuestras = "-".join([f"{m['codigoId']}:{m['nombreComun']}:{m['tipoUso']}:{m['nombreCientifico']}" for m in listaMuestras])

        cadenaRegistro = f"{idInspeccion}|{datosConglomerado}|{datosBrigada}|{conteoFustales}|{conteoLatizales}|{conteoBrinzales}|{cadenaMuestras}"

        self.capaDatos.guardarRegistro(cadenaRegistro)
        return True

    def buscarInspeccion(self, idBuscado):
        todosLosRegistros = self.capaDatos.leerTodosLosRegistros()
        for linea in todosLosRegistros:
            camposDatos = linea.strip().split('|')
            if len(camposDatos) >= 7 and camposDatos[0] == idBuscado:
                return {
                    "idInspeccion": camposDatos[0],
                    "datosConglomerado": camposDatos[1],
                    "datosBrigada": camposDatos[2],
                    "conteoFustales": camposDatos[3],
                    "conteoLatizales": camposDatos[4],
                    "conteoBrinzales": camposDatos[5],
                    "listaMuestras": camposDatos[6]
                }
        return None

# ==========================================
# CAPA DE PRESENTACIÓN (Presentation Layer)
# ==========================================
class CapaPresentacion:
    def __init__(self):
        self.capaNegocio = CapaNegocio()

    def mostrarMenuPrincipal(self):
        while True:
            print("\n--- INVENTARIO FORESTAL NACIONAL (IDEAM) ---")
            print("1. Registrar nueva inspección")
            print("2. Consultar inspección")
            print("3. Salir")

            opcionUsuario = input("Seleccione una opción: ")

            if opcionUsuario == "1":
                self.registrarNuevaInspeccion()
            elif opcionUsuario == "2":
                self.consultarInspeccionExistente()
            elif opcionUsuario == "3":
                print("Saliendo del sistema...")
                break
            else:
                print("Opción inválida. Intente de nuevo.")

    def registrarNuevaInspeccion(self):
        print("\n--- REGISTRO DE INSPECCIÓN ---")
        idInspeccion = input("ID de la inspección: ")
        datosConglomerado = input("Datos del conglomerado (Ubicación/Centroide): ")
        datosBrigada = input("Datos de la brigada encargada: ")

        try:
            conteoFustales = int(input("Cantidad de árboles Fustales: "))
            conteoLatizales = int(input("Cantidad de árboles Latizales: "))
            conteoBrinzales = int(input("Cantidad de árboles Brinzales: "))
        except ValueError:
            print("Error: El conteo de árboles debe ser numérico.")
            return

        listaMuestras = []
        agregarMasMuestras = "s"

        while agregarMasMuestras.lower() == "s":
            print("\n- Registro de Muestra -")
            codigoId = input("Código de identificación de la muestra: ")
            nombreComun = input("Nombre común de la especie: ")
            tipoUso = input("Uso (medicinal / alimenticio / maderable): ")
            nombreCientifico = input("Nombre científico (o 'pendiente' si va a laboratorio): ")

            listaMuestras.append({
                "codigoId": codigoId,
                "nombreComun": nombreComun,
                "tipoUso": tipoUso,
                "nombreCientifico": nombreCientifico
            })
            agregarMasMuestras = input("¿Desea agregar otra muestra? (s/n): ")

        self.capaNegocio.agregarInspeccion(
            idInspeccion, datosConglomerado, datosBrigada,
            conteoFustales, conteoLatizales, conteoBrinzales, listaMuestras
        )
        print("¡Inspección guardada exitosamente en el archivo de texto!")

    def consultarInspeccionExistente(self):
        print("\n--- CONSULTA DE INSPECCIÓN ---")
        idBuscado = input("Ingrese el ID de la inspección a consultar: ")
        resultadoInspeccion = self.capaNegocio.buscarInspeccion(idBuscado)

        if resultadoInspeccion:
            print("\n--- DATOS DE LA INSPECCIÓN ---")
            print(f"ID Inspección: {resultadoInspeccion['idInspeccion']}")
            print(f"Conglomerado: {resultadoInspeccion['datosConglomerado']}")
            print(f"Brigada: {resultadoInspeccion['datosBrigada']}")
            print(f"Conteo de árboles -> Fustales: {resultadoInspeccion['conteoFustales']}, Latizales: {resultadoInspeccion['conteoLatizales']}, Brinzales: {resultadoInspeccion['conteoBrinzales']}")

            print("\nMuestras recolectadas:")
            if resultadoInspeccion['listaMuestras']:
                arregloMuestras = resultadoInspeccion['listaMuestras'].split('-')
                for cadenaMuestra in arregloMuestras:
                    partesMuestra = cadenaMuestra.split(':')
                    if len(partesMuestra) == 4:
                        print(f"  - Código: {partesMuestra[0]} | Nombre Común: {partesMuestra[1]} | Uso: {partesMuestra[2]} | Científico: {partesMuestra[3]}")
            else:
                print("  No se registraron muestras.")
        else:
            print("No se encontró ninguna inspección registrada con ese ID.")

# ==========================================
# EJECUCIÓN PRINCIPAL
# ==========================================
if __name__ == "__main__":
    instanciaApp = CapaPresentacion()
    instanciaApp.mostrarMenuPrincipal()